# 03b — Carry with **Slosh** Payload (variable-mass ball inside a hollow box)

| | |
|---|---|
| **Author** | Narayane Ribeiro Medeiros |
| **Affiliation** | Instituto Tecnológico de Aeronáutica (ITA) |
| **Advisor** | Prof. Marcos Máximo |
| **Context** | Undergraduate's defense — 2026 |

Variant of `03a_carry_normal.ipynb`. Same MDP (goal reaching + twist tracking + hugging arms + walk-terminal reset), but the payload is physically embodied:

- Box → **hollow, semi-transparent** cube (6 wall geoms, α = 0.3).
- Ball → free-floating **solid sphere** with radius = `BOX_HALF / 4` spawned on the interior bottom.
- **Only the ball mass is randomised** (`BALL_MASS_MIN = 0.30 kg` → `BALL_MASS_MAX = 3.00 kg`). The box shell mass is fixed at `BOX_WALL_MASS_TOTAL = 0.20 kg`.

The ball injects internal disturbance torques as it slides against the walls, giving a more realistic domain-randomisation of "unknown payload dynamics" than lumped-mass variation on the box body itself.

Config lives at `config/policy_carry_slosh.py`; PPO logs go to `outputs/logs/g1_carry_slosh/`.

## Run-Mode Flags & Slosh Knobs

Boolean switches (reuse/resume/warm-start/viewer) plus slosh payload knobs:
number of balls, target interior fill, and the two slosh reward-weight terms
(quadratic velocity penalty + EMA COM-stability bonus).

In [ ]:
REUSE_TRAINING       = False    # skip training if PPO checkpoint exists
RESUME               = False    # continue from the most recent PPO checkpoint
WARM_START_FROM_WALK = True     # load walk checkpoint as initialization (hierarchical-style HRL)
DIAGNOSTIC_TRUNK_FROZEN = False # if True, freeze transferred trunk (mlp.2, mlp.4) and  replace head (mlp.6)
DEBUG_LIGHT          = True     # True = 64 envs / False = 8192 envs, few iters
RUN_VIEWER           = True     # open viser viewer at localhost:8080

# Slosh payload knobs: NUM_BALLS free spheres sized so their combined volume fills INTERIOR_FILL_PCT of the box interior; radius is auto-derived; per-startup total mass ~ U(0.30, 3.00) kg is split equally across balls.
NUM_BALLS         = 100
INTERIOR_FILL_PCT = 30.0
BALL_RADIUS_M     = 0.01

# Slosh reward weights: slosh penalty is QUADRATIC in ball-relative velocity; COM-stability is EXPONENTIAL over the EMA of the ball-cloud centroid.
SLOSH_PENALTY_WEIGHT   = -0.5
COM_STABILITY_WEIGHT   =  0.5
COM_STABILITY_SIGMA_M  = 0.03
COM_STABILITY_EMA_ALPHA = 0.05

# TODO: expose REMOVE_BOX_LID for open-top-carry experiments (drops box_top geom, raises slosh penalty, adds ball-outside-box termination).
# REMOVE_BOX_LID = False

## Payload preview

Uses the `NUM_BALLS` and `BALL_RADIUS_M` knobs from the previous cell to build the same hollow box + loose balls the training uses, drop it corner-down from 42 cm, and render 5 key frames of the fall + impact + settle. Runs in pure MuJoCo (no mjlab, no robot).

In [ ]:
import os
os.environ.setdefault('MUJOCO_GL', 'egl')

import math
import matplotlib.pyplot as plt
import mujoco
import numpy as np

_BOX_HALF, _BOX_WALL_T = 0.105, 0.008
_BOX_WALL_MASS_TOTAL, _TIMESTEP, _DROP_Z = 0.20, 0.001, 0.42
_TOTAL_PAYLOAD_MASS   = 1.5   # kg, middle of the DR range for illustration

# Corner-down rotation as an initial orientation quaternion (wxyz).
_theta = math.acos(-1.0 / math.sqrt(3.0))
_axis  = np.array([-1., 1., 0.]) / math.sqrt(2.0)
_qw = math.cos(_theta / 2.0); _qxyz = math.sin(_theta / 2.0) * _axis
_box_quat = (_qw, float(_qxyz[0]), float(_qxyz[1]), float(_qxyz[2]))

_spec = mujoco.MjSpec()
_spec.option.timestep = _TIMESTEP
_spec.option.integrator = mujoco.mjtIntegrator.mjINT_IMPLICITFAST
_spec.option.iterations = 15
_spec.option.ls_iterations = 20
_spec.nconmax, _spec.njmax = 400, 1600

_spec.worldbody.add_geom(
    name='floor', type=mujoco.mjtGeom.mjGEOM_PLANE, size=(2., 2., 0.1),
    rgba=(0.85, 0.85, 0.88, 1.0),
    solref=(0.002, 1.0), solimp=(0.98, 0.999, 1e-4, 0.5, 2.0),
)

# Hollow box (single body, 6 walls) placed at DROP_Z with the corner-down quat.
_inner = _BOX_HALF - _BOX_WALL_T
_wall_mass = _BOX_WALL_MASS_TOTAL / 6.0
_wall_common = dict(rgba=(0.85, 0.22, 0.18, 0.30), condim=6,
                    friction=(4.0, 0.25, 0.08), solref=(0.004, 1.0),
                    solimp=(0.95, 0.995, 1e-3, 0.5, 2.0), mass=_wall_mass)
_box = _spec.worldbody.add_body(name='box', pos=(0., 0., _DROP_Z), quat=_box_quat)
_box.add_freejoint().name = 'box_free'
_box.add_geom(name='box_bottom', type=mujoco.mjtGeom.mjGEOM_BOX,
              size=(_BOX_HALF, _BOX_HALF, _BOX_WALL_T), pos=(0., 0., -_inner), **_wall_common)
_box.add_geom(name='box_top',    type=mujoco.mjtGeom.mjGEOM_BOX,
              size=(_BOX_HALF, _BOX_HALF, _BOX_WALL_T), pos=(0., 0., +_inner), **_wall_common)
_side_yz = _BOX_HALF - 2 * _BOX_WALL_T
_box.add_geom(name='box_front', type=mujoco.mjtGeom.mjGEOM_BOX,
              size=(_BOX_WALL_T, _side_yz, _side_yz), pos=(+_inner, 0., 0.), **_wall_common)
_box.add_geom(name='box_back',  type=mujoco.mjtGeom.mjGEOM_BOX,
              size=(_BOX_WALL_T, _side_yz, _side_yz), pos=(-_inner, 0., 0.), **_wall_common)
_box.add_geom(name='box_left',  type=mujoco.mjtGeom.mjGEOM_BOX,
              size=(_side_yz, _BOX_WALL_T, _side_yz), pos=(0., +_inner, 0.), **_wall_common)
_box.add_geom(name='box_right', type=mujoco.mjtGeom.mjGEOM_BOX,
              size=(_side_yz, _BOX_WALL_T, _side_yz), pos=(0., -_inner, 0.), **_wall_common)

# Balls arranged on the interior bottom face; same corner-down quat rotates them with the box.
_k = int(math.ceil(math.sqrt(max(NUM_BALLS, 1))))
_inner_half = _BOX_HALF - 2 * _BOX_WALL_T
_span = 2.0 * (_inner_half - BALL_RADIUS_M)
_step = _span / (_k - 1) if _k > 1 else 0.0
_dz_floor = -_BOX_HALF + 2 * _BOX_WALL_T + BALL_RADIUS_M + 0.002
_per_ball_mass = _TOTAL_PAYLOAD_MASS / max(NUM_BALLS, 1)

# Rotation matrix from the box quaternion, used to place balls in world frame.
_R = np.array([
    [_qw*_qw + _qxyz[0]**2 - _qxyz[1]**2 - _qxyz[2]**2,
     2*(_qxyz[0]*_qxyz[1] - _qw*_qxyz[2]),
     2*(_qxyz[0]*_qxyz[2] + _qw*_qxyz[1])],
    [2*(_qxyz[0]*_qxyz[1] + _qw*_qxyz[2]),
     _qw*_qw - _qxyz[0]**2 + _qxyz[1]**2 - _qxyz[2]**2,
     2*(_qxyz[1]*_qxyz[2] - _qw*_qxyz[0])],
    [2*(_qxyz[0]*_qxyz[2] - _qw*_qxyz[1]),
     2*(_qxyz[1]*_qxyz[2] + _qw*_qxyz[0]),
     _qw*_qw - _qxyz[0]**2 - _qxyz[1]**2 + _qxyz[2]**2],
])

for i in range(NUM_BALLS):
    row, col = divmod(i, _k)
    local = np.array([
        -(_inner_half - BALL_RADIUS_M) + col * _step,
        -(_inner_half - BALL_RADIUS_M) + row * _step,
        _dz_floor,
    ])
    world = np.array([0., 0., _DROP_Z]) + _R @ local
    name = f'ball_{i}'
    b = _spec.worldbody.add_body(name=name, pos=tuple(world))
    b.add_freejoint().name = f'{name}_free'
    b.add_geom(name=f'{name}_g', type=mujoco.mjtGeom.mjGEOM_SPHERE,
               size=(BALL_RADIUS_M, 0., 0.), mass=_per_ball_mass,
               rgba=(1.0, 0.85, 0.10, 1.0),
               condim=6, friction=(1.5, 0.05, 0.02),
               solref=(0.004, 1.0), solimp=(0.95, 0.995, 1e-3, 0.5, 2.0))

_model = _spec.compile()
_data  = mujoco.MjData(_model)
mujoco.mj_forward(_model, _data)

print(f'Slosh payload preview      :  hollow cube + {NUM_BALLS} loose ball(s) inside')
print(f'  outer cube               :  {2*_BOX_HALF*100:.0f} × {2*_BOX_HALF*100:.0f} × {2*_BOX_HALF*100:.0f} cm')
print(f'  wall thickness           :  {2*_BOX_WALL_T*1000:.0f} mm  (α=0.30 semi-transparent)')
print(f'  interior cavity          :  {2*(_BOX_HALF - 2*_BOX_WALL_T)*100:.1f} × ... × ... cm')
print(f'  ball radius              :  {BALL_RADIUS_M*1000:.0f} mm  ({_k}×{_k} grid layout)')
print(f'  box shell mass           :  {_BOX_WALL_MASS_TOTAL:.2f} kg (fixed)')
print(f'  illustration payload     :  {_TOTAL_PAYLOAD_MASS:.2f} kg total  → {_per_ball_mass:.3f} kg per ball')

_W, _H = 720, 540
_model.vis.global_.offwidth  = max(_model.vis.global_.offwidth,  _W)
_model.vis.global_.offheight = max(_model.vis.global_.offheight, _H)
_ren = mujoco.Renderer(_model, height=_H, width=_W)
_cam = mujoco.MjvCamera()
_cam.lookat[:] = np.array([0., 0., 0.18])
_cam.distance, _cam.elevation, _cam.azimuth = 1.0, -18.0, 50.0

_frame_steps = (0, 150, 270, 340, 800)
_labels = ('initial', 'mid-fall', 'before impact', 'after impact', 'settled')
_frames = {}
for step in range(_frame_steps[-1] + 1):
    if step in _frame_steps:
        _ren.update_scene(_data, camera=_cam)
        _frames[step] = _ren.render().copy()
    mujoco.mj_step(_model, _data)

_fig, _axes = plt.subplots(1, 5, figsize=(20, 4))
for _ax, _step, _label in zip(_axes, _frame_steps, _labels):
    _ax.imshow(_frames[_step]); _ax.axis('off')
    _ax.set_title(f'{_label}  (t={_step*_TIMESTEP*1000:.0f} ms)', fontsize=11)
_fig.suptitle(
    f'Slosh payload — corner drop  |  NUM_BALLS={NUM_BALLS}   BALL_RADIUS_M={BALL_RADIUS_M*1000:.0f} mm',
    fontsize=12,
)
_fig.tight_layout()
plt.show()


## Environment Setup

Installs pinned dependencies from `config/requirements.txt` via **uv** (5-10x
faster than pip, idempotent), configures MuJoCo in headless mode (EGL), and
detects the training GPU.

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

if sys.version_info[:2] != (3, 10):
    raise RuntimeError(f'Python 3.10 required but got {sys.version.split()[0]}')

os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ['UV_HTTP_TIMEOUT'] = '600'

REQ = Path('../requirements.txt').resolve()
if not REQ.exists():
    raise FileNotFoundError(f'requirements.txt not found at {REQ}')

if shutil.which('uv') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])

subprocess.check_call([
    sys.executable, '-m', 'uv', 'pip', 'install',
    '--index-strategy', 'unsafe-best-match',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu128',
    '-r', str(REQ),
])

import torch, mjlab
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print(f'GPU count : {n}')
    for i in range(n):
        p = torch.cuda.get_device_properties(i)
        print(f'  [{i}] {p.name}  |  {p.total_memory / 1e9:.1f} GB  |  sm_{p.major}{p.minor}')
    DEVICE = 'cuda:0'
    torch.cuda.set_device(0)
else:
    DEVICE = 'cpu'
    print('WARNING: no GPU detected')

try:
    subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.free',
                    '--format=csv,noheader'], check=False)
except FileNotFoundError:
    pass


## Robot Initialization

Loads `TASK_ID`, G1 joints/home pose, and slosh-specific box/ball constants
(shell mass, ball total-mass DR range, goal range) from
`config/policy_carry_slosh.py`.

In [ ]:
import sys, os
from pathlib import Path
for _p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_p / 'LICENSE').exists():
        os.chdir(_p); break
sys.path.insert(0, str(Path.cwd()))

from src.policies.policy_carry_slosh import (
    TASK_ID,
    G1_JOINTS, G1_HOME_ROOT_POS, G1_HOME_ROOT_ROT, G1_HOME_JOINT_POS,
    BOX_HALF, BOX_HOLD_X_OFFSET, BOX_HOLD_Z, BOX_DROP_Z,
    BOX_WALL_T, BOX_WALL_MASS_TOTAL,
    BALL_TOTAL_MASS_MIN, BALL_TOTAL_MASS_MAX,
    GOAL_REACH_RADIUS, GOAL_DIST_STAGES,
    CHEST_Z,
)

print(f'Task ID           : {TASK_ID}')
print(f'Joints            : {len(G1_JOINTS)}  (derived from mjlab G1 spec)')
print(f'Home root z       : {G1_HOME_ROOT_POS[2]:.3f} m')
print()
print(f'Box (hollow)      : outer half-extent {BOX_HALF:.3f} m, wall half-thickness {BOX_WALL_T:.3f} m')
print(f'Box shell mass    : {BOX_WALL_MASS_TOTAL:.2f} kg (fixed — NOT randomised)')
print(f'Box start (front) : {BOX_HOLD_X_OFFSET:.2f} m in front of trunk, z={BOX_HOLD_Z:.2f} m')
print(f'Box drop limit    : z < {BOX_DROP_Z:.2f} m → termination (dropped)')
print()
print(f'Payload           : {NUM_BALLS} ball(s) of radius {BALL_RADIUS_M:.4f} m each')
print(f'Total payload DR  : [{BALL_TOTAL_MASS_MIN:.2f}, {BALL_TOTAL_MASS_MAX:.2f}] kg (split equally across balls)')
print(f'Per-ball mass DR  : [{BALL_TOTAL_MASS_MIN/NUM_BALLS:.3f}, {BALL_TOTAL_MASS_MAX/NUM_BALLS:.3f}] kg')
print()
print(f'Goal reach radius : {GOAL_REACH_RADIUS:.2f} m')
print(f'Goal dist stages  :')
for stg in GOAL_DIST_STAGES:
    print(f'    step ≥ {stg["step"]:>10,}  →  range = {stg["range"]} m')

## Task Config & Registration

MDP shared with `config/policy_carry.py` (goal, hugging-arm reset, walk-terminal reset, rewards); the slosh-specific overrides — **hollow box spec, ball entity, ball_mass startup DR, combined robot+box+ball reset** — live in **`config/policy_carry_slosh.py`**.

To tune ball dimensions, mass range, or box transparency, edit `config/policy_carry_slosh.py` and re-run this cell.

In [ ]:
from src.policies import policy_carry_slosh

# Purely geometric ball radius that packs INTERIOR_FILL_PCT of the interior with NUM_BALLS spheres; grid feasibility is checked below.
BALL_RADIUS_M = policy_carry_slosh.radius_from_fill_pct(NUM_BALLS, INTERIOR_FILL_PCT)
print(f'Derived ball radius: {BALL_RADIUS_M*1000:.2f} mm  '
      f'(N={NUM_BALLS}, fill={INTERIOR_FILL_PCT:.1f}%)')

# Prints the auto-computed 3D grid, pile height, headroom, actual fill; raises ValueError if the balls don't fit.
_fit = policy_carry_slosh.check_fits(num_balls=NUM_BALLS, ball_radius_m=BALL_RADIUS_M)

policy_carry_slosh.configure(num_balls=NUM_BALLS, ball_radius_m=BALL_RADIUS_M)
policy_carry_slosh.register()

# Override reward weights from the notebook-level knobs so tweaking them doesn't require editing the module.
_env = policy_carry_slosh.g1_carry_slosh_env_cfg(play=False)
_env.rewards['ball_slosh'].weight = SLOSH_PENALTY_WEIGHT
_env.rewards['ball_com_stability'].weight = COM_STABILITY_WEIGHT
_env.rewards['ball_com_stability'].params['sigma']     = COM_STABILITY_SIGMA_M
_env.rewards['ball_com_stability'].params['ema_alpha'] = COM_STABILITY_EMA_ALPHA

print(f'\nTask registered   : {policy_carry_slosh.TASK_ID}')
print(f'Entities          : {list(_env.scene.entities.keys())[:5]}… (+ {NUM_BALLS-4} balls)')
print(f"Sensors           : {len(_env.scene.sensors)}  ({', '.join(s.name for s in _env.scene.sensors)})")
print(f'Events            : {list(_env.events.keys())}')
print(f'  (note: box_mass replaced by ball_mass_split; reset_robot_and_box replaced by reset_robot_box_balls)')
print(f'Rewards           : {len(_env.rewards)} terms   → slosh={_env.rewards["ball_slosh"].weight}  '
      f'com_stab={_env.rewards["ball_com_stability"].weight}')
print(f'Slosh observations: {[t for t in _env.observations["actor"].terms if t.startswith("ball_")]}')
print(f'Terminations      : {list(_env.terminations.keys())}')
print(f'Curriculum        : {list(_env.curriculum.keys())}')
print(f'Sim iter/ls       : {_env.sim.mujoco.iterations}/{_env.sim.mujoco.ls_iterations}')
print(f'Sim nconmax/njmax : {_env.sim.nconmax}/{_env.sim.njmax}')
print(f'Episode length    : {_env.episode_length_s:.2f} s   ({int(_env.episode_length_s / 0.025)} steps @ 40 Hz)')


## Simulation Parameters

Global run settings shared by training and play: task, parallel envs, PPO
hyperparameters. `DEBUG_LIGHT` shrinks to 64 envs / 3 iters for a smoke test.

In [ ]:
from pathlib import Path
import sys; _root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'LICENSE').exists()); import os as _os; _os.chdir(_root); sys.path.insert(0, str(_root))
from src.utils import find_latest_checkpoint

OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
MEDIA = OUT / 'media'; MEDIA.mkdir(exist_ok=True)

EXPERIMENT_HINT      = 'g1_carry_slosh'
WALK_EXPERIMENT_HINT = 'g1_walk'  # source of the walk checkpoint used for warm-start
LOG_ROOT             = str(OUT / 'logs')

NUM_ENVS         = 8192
MAX_ITERATIONS   = 1000

SIM_TIMESTEP     = 0.005
DECIMATION       = 5
EPISODE_LENGTH_S = 8.0

LEARNING_RATE    = 3e-4
CLIP_PARAM       = 0.2
GAMMA            = 0.99
LAMBDA_GAE       = 0.95
ENTROPY_COEF     = 0.01

if DEBUG_LIGHT:
    NUM_ENVS, MAX_ITERATIONS = 64, 3

print(f'Task            : {TASK_ID}')
print(f'Parallel envs   : {NUM_ENVS}')
print(f'PPO iterations  : {MAX_ITERATIONS}')
print(f'Policy freq     : {1 / (SIM_TIMESTEP * DECIMATION):.0f} Hz')
print(f'Episode length  : {EPISODE_LENGTH_S:.2f} s')
print(f'Outputs dir     : {OUT.resolve()}')
print(f'Media dir       : {MEDIA.resolve()}')
print(f'Log root        : {Path(LOG_ROOT).resolve()}')


## Walk-Terminal Buffer

The carry reset (inherited from pick) samples each episode from a buffer of
walk-terminal states so the robot spawns in a walking-realistic pose and
velocity — critical for warm-start, because the transferred actor was trained
on that state distribution. Reuses `outputs/walk_terminal_states.npz` if
present, otherwise rolls the trained walk policy across parallel envs and
snapshots. Requires a walk checkpoint in `outputs/logs/g1_velocity/` (run
`01_walk.ipynb` first).

In [ ]:
import subprocess, sys
from pathlib import Path
_root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'LICENSE').exists()); import os as _os; _os.chdir(_root); sys.path.insert(0, str(_root))
from src.utils import find_latest_checkpoint
from src.policies.policy_carry import WALK_BUFFER_PATH

BUFFER_PATH   = Path(WALK_BUFFER_PATH).resolve()
BUFFER_N      = 20_000
BUFFER_ENVS   = 2048 if not DEBUG_LIGHT else 128

if BUFFER_PATH.exists():
    print(f'[reuse] walk buffer already at {BUFFER_PATH}')
else:
    walk_ckpt = find_latest_checkpoint(LOG_ROOT, experiment_hint='g1_walk')
    if walk_ckpt is None:
        raise RuntimeError(
            'No walk checkpoint found in outputs/logs/g1_walk. '
            'Train the walk policy first (01_walk.ipynb).'
        )
    print(f'Walk ckpt        : {walk_ckpt}')
    print(f'Collecting {BUFFER_N} walk-terminal states across {BUFFER_ENVS} envs...')
    subprocess.check_call([
        sys.executable, '-m', 'src.buffers.warm_buffer',
        '--checkpoint', str(walk_ckpt),
        '--out',        str(BUFFER_PATH),
        '--num-envs',   str(BUFFER_ENVS),
        '--n-snapshots', str(BUFFER_N),
    ])
    print(f'Buffer written   : {BUFFER_PATH}')

## Training Setup

Builds `TrainConfig` from the registered task and applies notebook overrides
(envs, iterations, logger). If `WARM_START_FROM_WALK=True`, the setup points
at the latest `g1_walk` checkpoint and uses the shape-lenient `load`
(`strict=False`, actor+critic only) so walk weights transfer to the first
carry-actor layers even when obs dimensionality changes (new dims start silent).

In [ ]:
import datetime, os
from pathlib import Path
from mjlab.scripts.train import TrainConfig

cfg = TrainConfig.from_task(TASK_ID)
cfg.env.scene.num_envs    = NUM_ENVS
cfg.agent.max_iterations  = MAX_ITERATIONS
cfg.agent.logger          = 'tensorboard'
cfg.agent.experiment_name = EXPERIMENT_HINT
cfg.agent.resume          = False

log_dir = (
    Path(LOG_ROOT) / EXPERIMENT_HINT
    / datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
)

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ.setdefault('MUJOCO_GL', 'egl')

_mode = 'RESUME' if RESUME else ('WARM_START_FROM_WALK' if WARM_START_FROM_WALK else 'FROM SCRATCH')
print(f'Task            : {TASK_ID}')
print(f'Envs / iters    : {cfg.env.scene.num_envs} / {cfg.agent.max_iterations}')
print(f'Logger          : {cfg.agent.logger}')
print(f'CUDA device     : {os.environ["CUDA_VISIBLE_DEVICES"]}')
print(f'Mode            : {_mode}')
print(f'Log dir         : {log_dir.resolve()}')


## Run Training

Blocks the kernel until PPO training finishes. Use "Interrupt Kernel" to stop.

In [ ]:
import sys, os
from pathlib import Path
for _p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_p / 'LICENSE').exists():
        os.chdir(_p); break
sys.path.insert(0, str(Path.cwd()))
from src.utils import find_latest_checkpoint
from src.training.train_loop import run as run_train_loop
from mjlab.utils.os import get_checkpoint_path

_existing_ckpt = find_latest_checkpoint(LOG_ROOT, experiment_hint=EXPERIMENT_HINT)
if REUSE_TRAINING and _existing_ckpt is not None:
    print(f'[REUSE] Checkpoint found — skipping training.\n        {_existing_ckpt}')
else:
    resume_ckpt, warm_ckpt = None, None
    if RESUME:
        resume_ckpt = get_checkpoint_path(
            log_dir.parent, r'\d{4}-\d{2}-\d{2}_.*', 'model_.*.pt',
        )
    elif WARM_START_FROM_WALK:
        warm_ckpt = find_latest_checkpoint(LOG_ROOT, experiment_hint=WALK_EXPERIMENT_HINT)
        if warm_ckpt is None:
            raise RuntimeError(
                f'WARM_START_FROM_WALK=True mas não achei checkpoint do walk em '
                f'{LOG_ROOT}/{WALK_EXPERIMENT_HINT}. Rode 01_walk.ipynb antes.'
            )

    run_train_loop(
        task_id=TASK_ID,
        cfg=cfg,
        log_dir=log_dir,
        warm_start_ckpt=warm_ckpt,
        resume_ckpt=resume_ckpt,
        diagnostic_trunk_frozen=DIAGNOSTIC_TRUNK_FROZEN,
    )


## Training Curves & Robustness

Post-training analysis in two blocks: (1) **training curves** parsed from the TensorBoard event file — reward, PPO diagnostics, per-term reward decomposition, terminations and command tracking; (2) **robustness** — headless rollouts of the trained policy under lateral pushes and a commanded-velocity sweep, following the RSL "Learning to Walk in Minutes" evaluation protocol (Rudin et al., CoRL 2022).

Set `RUN_ROBUSTNESS = False` in the setup cell below to skip figures 5–6 (the rollout-based ones).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

_root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'LICENSE').exists()); import os as _os; _os.chdir(_root); sys.path.insert(0, str(_root))
from src.utils import find_latest_tb_log, find_latest_checkpoint

RUN_ROBUSTNESS = True  # False skips Figs 5-6 (rollout-based)

COLORS = {
    'primary':   '#0072B2',
    'accent':    '#D55E00',
    'ok':        '#009E73',
    'warn':      '#E69F00',
    'neutral':   '#56B4E9',
    'text':      '#1A1A1A',
    'grid':      '#B0B0B0',
}

matplotlib.rcParams.update({
    'text.usetex': False,
    'font.family': 'serif',
    'font.serif': ['Times', 'Nimbus Roman', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'font.size': 9, 'axes.labelsize': 9, 'legend.fontsize': 7.5,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.linewidth': 0.6, 'axes.edgecolor': COLORS['text'],
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'xtick.top': True, 'ytick.right': True,
    'xtick.minor.visible': True, 'ytick.minor.visible': True,
    'grid.color': COLORS['grid'], 'grid.linewidth': 0.4, 'grid.alpha': 0.35,
    'figure.dpi': 120,
})

def load_scalar(log_dir, tag):
    """Load a scalar time-series from a TensorBoard event file.
    Args: log_dir — path to the TB log directory; tag — scalar tag name.
    Returns: tuple (steps, values) as numpy arrays (empty if tag missing).
    """
    ea = EventAccumulator(str(log_dir)); ea.Reload()
    if tag not in ea.Tags().get('scalars', []):
        return np.array([]), np.array([])
    evs = ea.Scalars(tag)
    return (np.array([e.step for e in evs], dtype=float),
            np.array([e.value for e in evs], dtype=float))

def list_scalars(log_dir, prefix=''):
    """List scalar tags available in a TensorBoard log, optionally filtered by prefix.
    Args: log_dir — path to the TB log directory; prefix — string filter.
    Returns: list of matching scalar tag names.
    """
    ea = EventAccumulator(str(log_dir)); ea.Reload()
    return [t for t in ea.Tags().get('scalars', []) if t.startswith(prefix)]

def smooth(x, w=15):
    """Apply a centered moving-average filter of window `w` to a 1-D signal.
    Args: x — input array; w — window length in samples.
    Returns: smoothed numpy array with the same length as `x`.
    """
    if len(x) < w or w < 2:
        return np.asarray(x, dtype=float)
    pad = w // 2
    xp = np.pad(x, pad, mode='edge')
    return np.convolve(xp, np.ones(w) / w, mode='valid')[:len(x)]

def apply_grid(ax):
    """Enable major/minor grid on a matplotlib axis (initial style).
    Args: ax — matplotlib Axes to style.
    Returns: none.
    """
    ax.grid(True, which='major', linewidth=0.5, alpha=0.45)
    ax.grid(True, which='minor', linewidth=0.3, alpha=0.60)
    for s in ax.spines.values():
        s.set_linewidth(0.6)
def apply_grid(ax):
    """Enable major/minor grid with auto minor locators (final override).
    Args: ax — matplotlib Axes to style.
    Returns: none.
    """
    ax.minorticks_on()
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.grid(True, which='major', linewidth=0.5, alpha=0.80)
    ax.grid(True, which='minor', linewidth=0.3, alpha=0.70)
    for s in ax.spines.values():
        s.set_linewidth(0.6)
TB_DIR = find_latest_tb_log(LOG_ROOT, experiment_hint=EXPERIMENT_HINT)
assert TB_DIR is not None, 'No TensorBoard log found — train first.'
print(f'TB dir : {TB_DIR}')


### Return & PPO Diagnostics

Plots training curves from the TensorBoard event file: mean reward, episode
length, reward gain dR/di, entropy, value loss, and policy std. Depends on
helpers defined above.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9.6, 5.2))

x_r, y_r = load_scalar(TB_DIR, 'Train/mean_reward')
if len(y_r):
    axes[0, 0].plot(x_r, smooth(y_r), color=COLORS['primary'], lw=1.2)
axes[0, 0].set_xlabel('PPO iteration')
axes[0, 0].set_ylabel('Mean episode reward')
apply_grid(axes[0, 0])

x, y = load_scalar(TB_DIR, 'Train/mean_episode_length')
if len(y):
    axes[0, 1].plot(x, smooth(y), color=COLORS['ok'], lw=1.2)
axes[0, 1].axhline(1000, ls=':', lw=0.6, color=COLORS['text'], alpha=0.6)
axes[0, 1].text(x[-1] if len(x) else 0, 1010, 'max horizon',
                ha='right', va='bottom', fontsize=7, color=COLORS['text'], alpha=0.7)
axes[0, 1].set_xlabel('PPO iteration')
axes[0, 1].set_ylabel('Mean episode length (steps)')
apply_grid(axes[0, 1])

if len(y_r) > 1:
    r_smooth = smooth(y_r)
    dr = np.gradient(r_smooth, x_r)
    axes[0, 2].plot(x_r, smooth(dr, w=25), color=COLORS['warn'], lw=1.2)
    axes[0, 2].axhline(0, ls=':', lw=0.6, color=COLORS['text'], alpha=0.6)
axes[0, 2].set_xlabel('PPO iteration')
axes[0, 2].set_ylabel(r'Reward gain $dR/di$')
apply_grid(axes[0, 2])

for ax, tag, ylabel, color, log in [
    (axes[1, 0], 'Loss/entropy',    r'Entropy $H(\pi)$',        COLORS['primary'], False),
    (axes[1, 1], 'Loss/value',      r'Value loss $L_V$',        COLORS['accent'],  True),
    (axes[1, 2], 'Policy/mean_std', r'Policy std $\sigma_\pi$', COLORS['ok'],      False),
]:
    x, y = load_scalar(TB_DIR, tag)
    if len(y):
        ax.plot(x, smooth(y), color=color, lw=1.2)
    ax.set_xlabel('PPO iteration')
    ax.set_ylabel(ylabel)
    if log:
        ax.set_yscale('log')
    apply_grid(ax)

fig.tight_layout()
plt.savefig(MEDIA / 'walk_fig1_training.png', dpi=150, bbox_inches='tight')
plt.show()

if len(y_r):
    print(f'Peak reward: {np.max(smooth(y_r)):.2f}')


## Run Viewer

Blocks the kernel while the viewer is open. Use "Interrupt Kernel" to stop.

In [ ]:
import sys, os
from pathlib import Path
for _p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_p / 'LICENSE').exists():
        os.chdir(_p); break
sys.path.insert(0, str(Path.cwd()))
from src.utils import find_latest_checkpoint
from mjlab.scripts.play import PlayConfig, run_play

PLAY_AGENT          = 'trained'   # @param ["trained","random","zero"]
PLAY_NUM_ENVS       = 1           # @param {type:"integer"}
PLAY_VIEWER         = 'viser'     # @param ["viser","native"]
CHECKPOINT_OVERRIDE = ''          # @param {type:"string"}  leave blank for latest

checkpoint = (
    Path(CHECKPOINT_OVERRIDE)
    if CHECKPOINT_OVERRIDE.strip()
    else find_latest_checkpoint(LOG_ROOT, experiment_hint=EXPERIMENT_HINT)
)

if PLAY_AGENT == 'trained' and checkpoint is None:
    raise RuntimeError('No checkpoint found. Train first or set CHECKPOINT_OVERRIDE.')

play_cfg = PlayConfig(
    agent=PLAY_AGENT,
    checkpoint_file=str(checkpoint) if (PLAY_AGENT == 'trained' and checkpoint) else None,
    num_envs=PLAY_NUM_ENVS,
    viewer=PLAY_VIEWER,
)

print(f'Agent      : {PLAY_AGENT}')
print(f'Viewer     : {PLAY_VIEWER}')
print(f'Num envs   : {PLAY_NUM_ENVS}')
if checkpoint:
    print(f'Checkpoint : {checkpoint}')


if RUN_VIEWER:
    print('Viewer starting — open http://localhost:8080 in your browser (viser).')
    print('Press Interrupt Kernel in the toolbar to close it.')
    run_play(TASK_ID, play_cfg)
else:
    print('Set RUN_VIEWER=True to open the trained policy.')